
# Phase 3 Experiment Plan — Multi-Model Scaffolding Benchmark

*Comparing LLM Behavioral Assistant Effects Across Models On ONET tasks*

## 1. Motivation

Prior GDPval experiments suggested that “assistant scaffolding” — a short, structured guidance message describing how to plan and self-review — can make model outputs more organized but not necessarily higher-quality.
To generalize those findings, this phase shifts from GDPval to a **task** from Alex's leaderboards, enabling comparison across a **broad set of frontier and open-source models** and direct validation against the leaderboards.

The goal is to determine whether scaffolding:

1. Improves average task quality (especially for weaker models), and
2. Produces rankings that more closely match established external leaderboards.

---

## 2. Task Prompt

A simple but realistic **planning-and-reasoning** task; pick "plan_menu_options" prompt from the leaderboard.

> **“Create a cheap and simple 7-day meal plan for me with my requirements and preferences.”**
> (Age 31, Weight 86.4 kg, Height 180 cm, Caucasian Australian; allergies = Shellfish, Gluten; intolerances = Lactose, Onion; likes = Meat, Potatoes, Mexican, Thai, Indian; dislikes = Very Salty, Very Sweet; note = Super-taster.)

---

## 3. Model Roles

| Role                               | Model                | Purpose                                                                               |
| ---------------------------------- | -------------------- | ------------------------------------------------------------------------------------- |
| **Assistant (scaffold generator)** | **GPT-4o**           | Produces one 250–300 word “three-phase workflow” note before task execution.          |
| **Workers (task performers)**      | 8 models (see below) | Generate the actual meal plans under plain vs scaffolded conditions.                  |
| **Evaluator (Grader)**             | **GPT-5**            | Scores each output 0–10 on clarity, suitability, cost-awareness, tone, and structure. |

**Worker Models**

| Provider    | Model ID                                                                              |
| ----------- | ------------------------------------------------------------------------------------- |
| OpenAI      | `gpt-3.5-turbo`  ·  `o3-mini-2025-01-31`  ·  `o4-mini-2025-04-16`  ·  `o3-2025-04-16` |
| Together AI | `openai:gpt-oss-120b`  ·  `deepseek-ai:DeepSeek-V3.1`                                 |
| Google      | `gemini-2.5-pro`  ·  `gemini-2.5-flash`                                               |

---

## 4. Experimental Design

| Factor              | Levels                        |
| ------------------- | ----------------------------- |
| **Task**            | 1 (Meal Plan)                 |
| **Condition**       | Plain (C₀) vs Scaffolded (C₁) |
| **Assistant Model** | GPT-4o (fixed)                |
| **Worker Models**   | 8                             |
| **Replicates**      | 3 per condition               |

**Total Runs = 8 × 2 × 3 = 48 outputs**

All completions will be stored as text and evaluated via GPT-5.

---

## 5. Assistant Model Scope

> The assistant serves as a *static mentor*.
> It delivers a pre-task “Three-Phase Workflow” outlining requirements checking, planning, and self-review steps.
> It **does not** rewrite or co-generate the worker’s answer.
> Its purpose is to prime the worker model into a reflective, professional frame of mind before execution — a lightweight form of **behavioral augmentation**, not collaboration.

---

## 6. Workflow Overview

```text
               ┌────────────────────────────┐
               │  Task Prompt (Meal Plan)   │
               └────────────┬───────────────┘
                            │
        ┌───────────────────┴───────────────────┐
        │                                       │
 ┌──────▼──────┐                        ┌───────▼────────┐
 │ Assistant   │                        │ Worker Model   │
 │ (GPT-4o)    │                        │ (various)      │
 │ generates   │                        │ receives plain │
 │ scaffold    │                        │ or scaffolded  │
 │ guidance    │                        │ prompt         │
 └──────┬──────┘                        └───────┬────────┘
        │                                       │
        │               Outputs                 │
        └───────────────► Meal-Plan Text ◄──────┘
                            │
                            ▼
                 ┌───────────────────────┐
                 │ Evaluator (GPT-5)     │
                 │ Scores 0–10 + feedback│
                 └───────────┬───────────┘
                             ▼
                   Analysis / Ranking / ρ
```

---

## 7. Evaluation Methodology

1. **LLM Scoring:** GPT-5 assigns each response a 0–10 score + short justification.
2. **Averaging:** Mean ± SD per model × condition.
3. **Δ Effect:** (C₁ − C₀) per model.
4. **Leaderboard Comparison:** between our mean scores and the Stanford Leaderboard ranks.
5. **Interpretation:** Examine whether scaffolding raises weaker models, narrows rank gaps, and increases correlation with external performance order.

---
---



In [ ]:
!pip install edsl

# Augmentation: Fixing base model; vary scaffold models

Base: gpt-3.5 turbo, Scaffold: various models
Evaluator: gpt-5

Task: plan menu options

See if consistent with leaderboards

In [ ]:
from edsl import Model
Model.available()

In [ ]:
from edsl import login
login()

In [ ]:
# =========================================================
# Varying Assistant Scaffolds (Fixed Worker)
# =========================================================
from edsl import Agent, Model, Scenario, ScenarioList, Survey, QuestionFreeText
import pandas as pd
from pathlib import Path

# ---------------------------------------------------------
# 1. Experiment Configuration
# ---------------------------------------------------------
BASE_WORKER_MODEL = "gpt-3.5-turbo"   # fixed base model (worker)
EVALUATOR_MODEL = "gpt-5"             # for later grading
N_REPLICATES = 3                    # replicates per condition per assistant

ASSISTANT_MODELS = {
    "gpt-4.1": "GPT-4.1",
    "deepseek-ai/DeepSeek-V3.1": "DeepSeek-V3.1",
    "o4-mini-2025-04-16": "GPT-O4-Mini",
    "o3-mini-2025-01-31": "GPT-O3-Mini",
    "openai/gpt-oss-120b": "GPT-OSS-120B",
    "anthropic.claude-3-5-sonnet-20240620-v1:0": "Claude-3.5-Sonnet",
    "gpt-5.4-2026-03-05": "GPT-5.4",                          # verify via Model.available()
    "claude-opus-4-6": "Claude-Opus-4.6",    # verify via Model.available()
    "gemini-3.1-pro-preview": "Gemini-3.1-Pro",     # verify via Model.available()
}

# ---------------------------------------------------------
# 2–7: Everything below is UNCHANGED from your code
# ---------------------------------------------------------
TASK_NAME = "meal_plan"
TASK_PROMPT = """
Create a cheap and simple 7-day meal plan for me with my requirements and preferences.
Age: 31 Years
Current Weight: 86.4 Kilograms
Height: 180 Centimeters
Ethnicity: Caucasian Australian
Allergies: Shellfish, Gluten
Intolerances: Lactose, Onion
Likes: Meat, Potatoes, Mexican, Thai, Indian
Dislikes: Very Salty, Very Sweet
Notes: I have the super-tasting gene.
"""

ASSISTANT_PROMPT_TEMPLATE = """
You are a planning coach helping another model prepare to complete a task.

Your role is to provide PROCESS guidance only, not task content.

Write a short 200–250 word guidance message that gives a three-phase workflow:

1. Requirements Check
- identify the deliverable, audience, tone, and constraints
- note what a strong answer must include

2. Plan & Structure
- suggest a general response structure or sequence of sections
- suggest how to balance completeness, clarity, and tone

3. Draft -> Self-Review -> Finalize
- suggest a brief checklist for revising the answer before submission

Important rules:
- Do NOT solve the task
- Do NOT provide topic-specific content, examples, or recommendations
- Do NOT write sentences that could be copied into the final answer
- Do NOT restate the task as a completed outline with filled-in content
- Keep the guidance generic and process-focused

Format in Markdown under the heading:
**Assistant Guidance: Three-Phase Workflow**
"""

def generate_scaffold(task_prompt, assistant_model):
    assistant_agent = Agent(instruction=ASSISTANT_PROMPT_TEMPLATE)
    q = QuestionFreeText("scaffold", "{{ scenario.task_prompt }}")
    survey = Survey([q])
    scenario = Scenario({"task_prompt": task_prompt})
    print(f"\n🧩 Generating scaffold from {assistant_model}...")
    results = survey.by(scenario).by(assistant_agent).by(Model(assistant_model)).run()
    scaffold_text = results.select("scaffold").to_list()[0]
    print(f"✅ Scaffold generated from {assistant_model[:40]}...")
    return scaffold_text

def run_condition(base_worker_model, condition, task_prompt, scaffold_text=None):
    full_prompt = f"{scaffold_text}\n\n{task_prompt}" if scaffold_text else task_prompt
    scenarios = ScenarioList([
        Scenario({
            "task_prompt": full_prompt,
            "condition": condition,
            "replicate_id": i
        })
        for i in range(N_REPLICATES)
    ])
    q = QuestionFreeText("output", "{{ scenario.task_prompt }}")
    survey = Survey([q])
    worker = Agent(
        instruction=(
            "If the task prompt includes an 'Assistant Guidance: Three-Phase Workflow' section at the top, "
            "read it carefully first and follow its planning and self-review steps before producing your response. "
            "Then complete the task itself clearly and professionally. "
            "Return only the final deliverable (the completed meal plan) — do not restate the guidance."
        )
    )
    print(f"🚀 Running {base_worker_model} [{condition}]...")
    results = (
        survey.by(scenarios)
        .by(worker)
        .by(Model(base_worker_model))
        .run(
            n=1, progress_bar=True, verbose=True,
            remote_inference_description=f"{TASK_NAME}-{base_worker_model}-{condition}",
            remote_inference_results_visibility="private",
        )
    )
    df = results.select("scenario.replicate_id", "answer.output", "scenario.condition").to_pandas()
    df.rename(columns={
        "scenario.replicate_id": "replicate_id",
        "answer.output": "output",
        "scenario.condition": "condition",
    }, inplace=True)
    df["assistant_model"] = condition.replace("scaffold_", "")
    df["worker_model"] = base_worker_model
    safe_name = condition.replace(":", "_").replace("/", "_")
    Path(f"phase3b-outputs/{TASK_NAME}/{safe_name}").mkdir(parents=True, exist_ok=True)
    df.to_csv(f"phase3b-outputs/{TASK_NAME}/{safe_name}/outputs.csv", index=False)
    print(f"💾 Saved {condition} responses ({len(df)} rows)")
    return df

all_results = []
plain_df = run_condition(BASE_WORKER_MODEL, "plain", TASK_PROMPT)
all_results.append(plain_df)

for model_id, model_label in ASSISTANT_MODELS.items():
    scaffold_text = generate_scaffold(TASK_PROMPT, model_id)
    condition = f"scaffold_{model_label.replace(' ', '_')}"
    scaffold_df = run_condition(BASE_WORKER_MODEL, condition, TASK_PROMPT, scaffold_text=scaffold_text)
    all_results.append(scaffold_df)

final_df = pd.concat(all_results, ignore_index=True)
Path("phase3b-results").mkdir(exist_ok=True)
output_path = "phase3b-results/mealplan_vary_scaffolds.csv"
final_df.to_csv(output_path, index=False)
print(f"\n✅ Exported all results → {output_path} ({len(final_df)} rows)")
print(final_df.head())


## pairwise grading

In [ ]:
# ======================================================
# Phase 3b Evaluation: Pairwise Grading + Average Rank
# ======================================================

from edsl import Agent, Model, Scenario, ScenarioList, Survey, QuestionMultipleChoice
import pandas as pd
import numpy as np
import itertools
from pathlib import Path

# ------------------------------------------------------
# Config
# ------------------------------------------------------
INPUT_FILE = "phase3b-results/mealplan_vary_scaffolds.csv"
OUTPUT_FILE = "phase3b-results/mealplan_pairwise_ranked.csv"
LEADERBOARD_FILE = "phase3b-results/mealplan_pairwise_leaderboard.csv"

# Use one of the supported default models
EVALUATOR_MODEL = "gpt-5"
N_EVALS = 3  # independent pairwise judgments per pair

# ------------------------------------------------------
# Task Context
# ------------------------------------------------------
TASK_TEXT = """
Create a cheap and simple 7-day meal plan for me with my requirements and preferences.
Age: 31 Years
Current Weight: 86.4 Kilograms
Height: 180 Centimeters
Ethnicity: Caucasian Australian
Allergies: Shellfish, Gluten
Intolerances: Lactose, Onion
Likes: Meat, Potatoes, Mexican, Thai, Indian
Dislikes: Very Salty, Very Sweet
Notes: I have the super-tasting gene.
"""

# ------------------------------------------------------
# Pairwise Evaluation Prompt
# ------------------------------------------------------
EVAL_PROMPT = """
You are an expert evaluator simulating a thoughtful human reviewing practical work.

You will receive:
1. The original meal-plan request
2. Two candidate meal plans

Choose the one that is better overall for actual use.

Consider silently:
- Does it respect dietary restrictions (gluten, lactose, onion, shellfish)?
- Is it realistic, affordable, and simple?
- Is it organized and easy to follow?
- Which one would a real person be more likely to use successfully?

Return only:
- option_1
or
- option_2

No reasoning. No extra text.
"""

# ------------------------------------------------------
# Helpers
# ------------------------------------------------------
def build_pairwise_scenarios(df, n_evals):
    scenarios = []
    item_pairs = list(itertools.combinations(df.index.tolist(), 2))

    for i, j in item_pairs:
        row_i = df.loc[i]
        row_j = df.loc[j]

        for replicate_id in range(n_evals):
            scenarios.append(
                Scenario(
                    {
                        "left_idx": i,
                        "right_idx": j,
                        "replicate_id": replicate_id,
                        "task_text": TASK_TEXT,
                        "option_1": str(row_i["output"]),
                        "option_2": str(row_j["output"]),
                        "left_worker_model": row_i.get("worker_model", "unknown"),
                        "right_worker_model": row_j.get("worker_model", "unknown"),
                        "left_assistant_model": row_i.get("assistant_model", "none"),
                        "right_assistant_model": row_j.get("assistant_model", "none"),
                        "left_condition": row_i.get("condition", "unknown"),
                        "right_condition": row_j.get("condition", "unknown"),
                    }
                )
            )

    return ScenarioList(scenarios)


def evaluate_pairwise(df, n_evals=N_EVALS):
    df = df.copy().reset_index(drop=True)

    print(f"📂 Loaded {len(df)} candidate outputs")
    print(f"⚖️  Building all pairwise comparisons × {n_evals} replicates")

    scenarios = build_pairwise_scenarios(df, n_evals=n_evals)

    q = QuestionMultipleChoice(
        question_name="winner",
        question_text="""
Original request:
{{ task_text }}

--------------------------------------------------
OPTION 1
{{ option_1 }}

--------------------------------------------------
OPTION 2
{{ option_2 }}

Which option is better overall?
""",
        question_options=["option_1", "option_2"],
        include_comment=False,
    )

    survey = Survey([q])
    agent = Agent(instruction=EVAL_PROMPT)
    evaluator = Model(EVALUATOR_MODEL)

    results = (
        survey
        .by(scenarios)
        .by(agent)
        .by(evaluator)
        .run(
            progress_bar=False,
            verbose=False,
            print_exceptions=True,
            stop_on_exception=False,
        )
    )

    pairwise_df = results.select(
        "left_idx",
        "right_idx",
        "replicate_id",
        "winner",
    ).to_pandas()

    # Normalize column names in case EDSL prefixes them
    pairwise_df.columns = [c.split(".")[-1] for c in pairwise_df.columns]

    return df, pairwise_df


def aggregate_rankings(df, pairwise_df, n_evals=N_EVALS):
    records = []

    # One tournament per replicate_id
    for replicate_id in range(n_evals):
        sub = pairwise_df[pairwise_df["replicate_id"] == replicate_id].copy()

        wins = {i: 0 for i in df.index}
        games = {i: 0 for i in df.index}

        for _, row in sub.iterrows():
            left = int(row["left_idx"])
            right = int(row["right_idx"])
            winner = str(row["winner"]).strip()

            games[left] += 1
            games[right] += 1

            if winner == "option_1":
                wins[left] += 1
            elif winner == "option_2":
                wins[right] += 1

        rep_df = pd.DataFrame(
            {
                "item_id": list(df.index),
                "replicate_id": replicate_id,
                "wins": [wins[i] for i in df.index],
                "games": [games[i] for i in df.index],
            }
        )
        rep_df["win_rate"] = rep_df["wins"] / rep_df["games"].replace(0, np.nan)
        rep_df["rank"] = rep_df["wins"].rank(ascending=False, method="average")

        records.append(rep_df)

    long_rank_df = pd.concat(records, ignore_index=True)

    summary = (
        long_rank_df.groupby("item_id")
        .agg(
            avg_rank=("rank", "mean"),
            std_rank=("rank", "std"),
            avg_win_rate=("win_rate", "mean"),
            std_win_rate=("win_rate", "std"),
            total_wins=("wins", "sum"),
            total_games=("games", "sum"),
        )
        .reset_index()
    )

    scored = df.copy()
    scored["item_id"] = scored.index
    scored = scored.merge(summary, on="item_id", how="left")

    # Lower avg_rank is better
    scored = scored.sort_values(["avg_rank", "avg_win_rate"], ascending=[True, False])

    return scored, long_rank_df


# ======================================================
# Main
# ======================================================
def main():
    df = pd.read_csv(INPUT_FILE)

    scored_base, pairwise_df = evaluate_pairwise(df, n_evals=N_EVALS)
    scored, long_rank_df = aggregate_rankings(scored_base, pairwise_df, n_evals=N_EVALS)

    Path("phase3b-results").mkdir(exist_ok=True)

    scored.to_csv(OUTPUT_FILE, index=False)
    print(f"✅ Saved pairwise-ranked results → {OUTPUT_FILE}")

    leaderboard = (
        scored.groupby(["worker_model", "assistant_model", "condition"])
        .agg(
            avg_rank=("avg_rank", "mean"),
            avg_win_rate=("avg_win_rate", "mean"),
            std_rank=("avg_rank", "std"),
        )
        .sort_values(by=["avg_rank", "avg_win_rate"], ascending=[True, False])
        .reset_index()
    )

    leaderboard.to_csv(LEADERBOARD_FILE, index=False)
    print(f"🏁 Leaderboard saved → {LEADERBOARD_FILE}")

    print("\nTop rows:")
    print(
        scored[
            [
                "worker_model",
                "assistant_model",
                "condition",
                "avg_rank",
                "std_rank",
                "avg_win_rate",
                "total_wins",
                "total_games",
            ]
        ].head(10)
    )

    print("\nLeaderboard:")
    print(leaderboard.head(10))


if __name__ == "__main__":
    main()

In [ ]:
ranked = pd.read_csv("phase3b-results/mealplan_pairwise_ranked.csv")
print(ranked[["assistant_model", "condition", "replicate_id", "avg_rank", "avg_win_rate", "output"]].head(5))
print(ranked[["assistant_model", "condition", "replicate_id", "avg_rank", "avg_win_rate", "output"]].tail(5))

In [ ]:
# ============================================================
# 📊 Phase 3b Visualization — Scaffold Comparison (Win Rate)
# ============================================================
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# --- Load leaderboard ---
df = pd.read_csv("phase3b-results/mealplan_pairwise_leaderboard.csv")

# --- Normalize assistant names to match ASSISTANT_MODELS labels ---
assistant_name_map = {
    "gpt-4.1": "GPT-4.1",
    "deepseek-ai/DeepSeek-V3.1": "DeepSeek-V3.1",
    "o4-mini-2025-04-16": "GPT-O4-Mini",
    "o3-mini-2025-01-31": "GPT-O3-Mini",
    "openai/gpt-oss-120b": "GPT-OSS-120B",
    "anthropic.claude-3-5-sonnet-20240620-v1:0": "Claude-3.5-Sonnet",
    "gpt-5.4": "GPT-5.4",
    "claude-opus-4-6": "Claude-Opus-4.6",
    "gemini-3.1-pro-preview": "Gemini-3.1-Pro",
    "plain": "No Scaffold",
}

df["assistant_model"] = df["assistant_model"].replace(assistant_name_map)

# --- Clean up condition for legend/grouping ---
df["condition"] = np.where(
    df["condition"] == "plain",
    "Plain",
    "Scaffolded"
)

# --- Optional: enforce plotting order to match ASSISTANT_MODELS ---
model_order = [
    "No Scaffold",
    "GPT-4.1",
    "DeepSeek-V3.1",
    "GPT-O4-Mini",
    "GPT-O3-Mini",
    "GPT-OSS-120B",
    "Claude-3.5-Sonnet",
    "GPT-5.4",
    "Claude-Opus-4.6",
    "Gemini-3.1-Pro",
]

df["assistant_model"] = pd.Categorical(
    df["assistant_model"],
    categories=model_order,
    ordered=True
)

# --- Convert win rate to percent for plotting ---
df["win_rate_pct"] = df["avg_win_rate"] * 100
df["std_win_rate_pct"] = df["std_win_rate"] * 100 if "std_win_rate" in df.columns else 0.0

# --- Sort for display ---
df = df.sort_values(by="assistant_model")

# --- Seaborn style ---
sns.set(style="whitegrid", context="talk", font_scale=1.1)
plt.figure(figsize=(14, 7))

# --- Barplot ---
ax = sns.barplot(
    data=df,
    x="assistant_model",
    y="win_rate_pct",
    hue="condition",
    palette={"Plain": "#4C72B0", "Scaffolded": "#55A868"},
)

# --- Add custom error bars manually using std_win_rate ---
for i, bar in enumerate(ax.patches):
    if i < len(df):
        row = df.iloc[i]
        x = bar.get_x() + bar.get_width() / 2
        y = bar.get_height()
        plt.errorbar(
            x=x,
            y=y,
            yerr=row["std_win_rate_pct"],
            fmt="none",
            ecolor="black",
            elinewidth=1.2,
            capsize=4
        )

# --- Add value labels above each bar ---
for bar, (_, row) in zip(ax.patches, df.iterrows()):
    y = bar.get_height()
    x = bar.get_x() + bar.get_width() / 2
    plt.text(
        x,
        y + 1.0,
        f"{row['win_rate_pct']:.1f}%±{row['std_win_rate_pct']:.1f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

# --- Labels, title, legend ---
plt.title(
    "Impact of Assistant Scaffolds on GPT-3.5 Turbo Performance\n"
    "(Mean Pairwise Win Rate ± 1 SD)",
    fontsize=17,
    weight="bold"
)
plt.ylabel("Average Pairwise Win Rate (%)", fontsize=13)
plt.xlabel("Assistant Scaffold Model", fontsize=12)
plt.xticks(rotation=30, ha="right", fontsize=11)
plt.ylim(0, 100)
plt.legend(title="Condition", title_fontsize=12, fontsize=11)
plt.tight_layout()

# --- Save & show ---
plt.savefig("phase3b_scaffold_winrate.png", dpi=300, bbox_inches="tight")
plt.show()


# Automation scheme

In [ ]:
# =========================================================
# Direct Model Comparison (Each Model Does the Task Itself)
# =========================================================
from edsl import Agent, Model, Scenario, ScenarioList, Survey, QuestionFreeText
import pandas as pd
from pathlib import Path

# ---------------------------------------------------------
# 1. Experiment Configuration
# ---------------------------------------------------------
N_REPLICATES = 3

TASK_MODELS = {
    "gpt-3.5-turbo": "GPT-3.5-Turbo",
    "gpt-4.1": "GPT-4.1",
    "deepseek-ai/DeepSeek-V3.1": "DeepSeek-V3.1",
    "o4-mini-2025-04-16": "GPT-O4-Mini",
    "o3-mini-2025-01-31": "GPT-O3-Mini",
    "openai/gpt-oss-120b": "GPT-OSS-120B",
    "anthropic.claude-3-5-sonnet-20240620-v1:0": "Claude-3.5-Sonnet",
    "gpt-5.4": "GPT-5.4",
    "claude-opus-4-6": "Claude-Opus-4.6",
    "gemini-3.1-pro-preview": "Gemini-3.1-Pro",
}

# ---------------------------------------------------------
# 2. Task
# ---------------------------------------------------------
TASK_NAME = "meal_plan"
TASK_PROMPT = """
Create a cheap and simple 7-day meal plan for me with my requirements and preferences.
Age: 31 Years
Current Weight: 86.4 Kilograms
Height: 180 Centimeters
Ethnicity: Caucasian Australian
Allergies: Shellfish, Gluten
Intolerances: Lactose, Onion
Likes: Meat, Potatoes, Mexican, Thai, Indian
Dislikes: Very Salty, Very Sweet
Notes: I have the super-tasting gene.
"""

# ---------------------------------------------------------
# 3. Run one model directly on the task
# ---------------------------------------------------------
def run_model_condition(model_id, model_label, task_prompt):
    scenarios = ScenarioList([
        Scenario({
            "task_prompt": task_prompt,
            "condition": model_label,
            "replicate_id": i
        })
        for i in range(N_REPLICATES)
    ])

    q = QuestionFreeText("output", "{{ scenario.task_prompt }}")
    survey = Survey([q])

    worker = Agent(
        instruction=(
            "Complete the task clearly and professionally. "
            "Return only the final deliverable."
        )
    )

    print(f"🚀 Running {model_id} [{model_label}]...")

    results = (
        survey.by(scenarios)
        .by(worker)
        .by(Model(model_id, max_tokens=4096))
        .run(
            n=1,
            progress_bar=False,
            verbose=False,
            print_exceptions=True,
            stop_on_exception=False,
            check_api_keys=False,
            remote_inference_description="meal plan comparison"

        )
    )

    df = results.select(
        "scenario.replicate_id",
        "answer.output",
        "scenario.condition"
    ).to_pandas()

    df.rename(columns={
        "scenario.replicate_id": "replicate_id",
        "answer.output": "output",
        "scenario.condition": "condition",
    }, inplace=True)

    df["model_id"] = model_id
    df["model_label"] = model_label

    safe_name = model_label.replace(" ", "_").replace(":", "_").replace("/", "_")
    Path(f"phase3b-outputs/automation/{TASK_NAME}/{safe_name}").mkdir(parents=True, exist_ok=True)
    df.to_csv(f"phase3b-outputs/automation/{TASK_NAME}/{safe_name}/outputs.csv", index=False)

    print(f"💾 Saved {model_label} responses ({len(df)} rows)")
    return df
# ---------------------------------------------------------
# 4. Main
# ---------------------------------------------------------
all_results = []

for model_id, model_label in TASK_MODELS.items():
    model_df = run_model_condition(model_id, model_label, TASK_PROMPT)
    all_results.append(model_df)

final_df = pd.concat(all_results, ignore_index=True)

Path("phase3b-results").mkdir(exist_ok=True)
output_path = "phase3b-results/mealplan_automation_models.csv"
final_df.to_csv(output_path, index=False)

print(f"\n✅ Exported all results → {output_path} ({len(final_df)} rows)")
print(final_df[["replicate_id", "condition", "model_label", "output"]].head())

In [ ]:
# ======================================================
# Phase 3b Evaluation: Automation Models Pairwise Grading
# ======================================================

from edsl import Agent, Model, Scenario, ScenarioList, Survey, QuestionMultipleChoice
import pandas as pd
import numpy as np
import itertools
from pathlib import Path

# ------------------------------------------------------
# Config
# ------------------------------------------------------
INPUT_FILE = "phase3b-results/mealplan_automation_models.csv"
OUTPUT_FILE = "phase3b-results/mealplan_automation_pairwise_ranked.csv"
LEADERBOARD_FILE = "phase3b-results/mealplan_automation_pairwise_leaderboard.csv"

EVALUATOR_MODEL_NAME = "gpt-5"
N_EVALS = 3 # number of independent pairwise judgments per pair

# ------------------------------------------------------
# Task Context
# ------------------------------------------------------
TASK_TEXT = """
Create a cheap and simple 7-day meal plan for me with my requirements and preferences.
Age: 31 Years
Current Weight: 86.4 Kilograms
Height: 180 Centimeters
Ethnicity: Caucasian Australian
Allergies: Shellfish, Gluten
Intolerances: Lactose, Onion
Likes: Meat, Potatoes, Mexican, Thai, Indian
Dislikes: Very Salty, Very Sweet
Notes: I have the super-tasting gene.
"""

# ------------------------------------------------------
# Pairwise Evaluation Prompt
# ------------------------------------------------------
EVAL_PROMPT = """
You are an expert evaluator simulating a thoughtful human reviewing a practical work result.

You will receive:
1. The original meal-plan request
2. Two candidate meal plans

Judge as if you were a real person trying to use them.

Consider silently:
- Does it respect dietary restrictions (gluten, lactose, onion, shellfish)?
- Is it realistic, affordable, and simple?
- Is it organized and easy to follow?
- Which one would be more useful in practice?

Output only one of:
- option_1
- option_2

No reasoning. No extra text.
"""

# ------------------------------------------------------
# Helpers
# ------------------------------------------------------
def build_pairwise_scenarios(df, n_evals):
    scenarios = []
    item_pairs = list(itertools.combinations(df.index.tolist(), 2))

    for i, j in item_pairs:
        row_i = df.loc[i]
        row_j = df.loc[j]

        if pd.isna(row_i["output"]) or pd.isna(row_j["output"]):
            continue

        for replicate_id in range(n_evals):
            scenarios.append(
                Scenario(
                    {
                        "left_idx": int(i),
                        "right_idx": int(j),
                        "replicate_id": int(replicate_id),
                        "task_text": str(TASK_TEXT),
                        "option_1": str(row_i["output"]),
                        "option_2": str(row_j["output"]),
                        "left_model_id": str(row_i.get("model_id", "unknown")),
                        "right_model_id": str(row_j.get("model_id", "unknown")),
                        "left_model_label": str(row_i.get("model_label", "unknown")),
                        "right_model_label": str(row_j.get("model_label", "unknown")),
                        "left_condition": str(row_i.get("condition", "unknown")),
                        "right_condition": str(row_j.get("condition", "unknown")),
                        "left_replicate_id": int(row_i["replicate_id"]) if pd.notna(row_i.get("replicate_id")) else None,
                        "right_replicate_id": int(row_j["replicate_id"]) if pd.notna(row_j.get("replicate_id")) else None,
                    }
                )
            )

    return ScenarioList(scenarios)


def evaluate_pairwise(df, n_evals=N_EVALS):
    df = df.copy().reset_index(drop=True)
    df = df[df["output"].notna()].reset_index(drop=True)

    print(f"Loaded {len(df)} candidate outputs")
    print(f"Building all pairwise comparisons x {n_evals} replicates")

    scenarios = build_pairwise_scenarios(df, n_evals=n_evals)

    q = QuestionMultipleChoice(
        question_name="winner",
        question_text="""
Original request:
{{ task_text }}

--------------------------------------------------
OPTION 1
{{ option_1 }}

--------------------------------------------------
OPTION 2
{{ option_2 }}

Which option is better overall?
""",
        question_options=["option_1", "option_2"],
        include_comment=False,
    )

    survey = Survey([q])
    agent = Agent(instruction=EVAL_PROMPT)
    evaluator = Model(EVALUATOR_MODEL_NAME)

    results = (
        survey.by(scenarios)
        .by(agent)
        .by(evaluator)
        .run(
            n=1,
            progress_bar=False,
            verbose=False,
            stop_on_exception=False,
            check_api_keys=False,
            print_exceptions=True,
        )
    )

    pairwise_df = results.select(
        "left_idx",
        "right_idx",
        "replicate_id",
        "winner",
    ).to_pandas()

    pairwise_df.columns = [c.split(".")[-1] for c in pairwise_df.columns]

    return df, pairwise_df


def aggregate_rankings(df, pairwise_df, n_evals=N_EVALS):
    records = []

    for replicate_id in range(n_evals):
        sub = pairwise_df[pairwise_df["replicate_id"] == replicate_id].copy()

        wins = {i: 0 for i in df.index}
        games = {i: 0 for i in df.index}

        for _, row in sub.iterrows():
            left = int(row["left_idx"])
            right = int(row["right_idx"])
            winner = str(row["winner"]).strip()

            games[left] += 1
            games[right] += 1

            if winner == "option_1":
                wins[left] += 1
            elif winner == "option_2":
                wins[right] += 1

        rep_df = pd.DataFrame(
            {
                "item_id": list(df.index),
                "replicate_id": replicate_id,
                "wins": [wins[i] for i in df.index],
                "games": [games[i] for i in df.index],
            }
        )
        rep_df["win_rate"] = rep_df["wins"] / rep_df["games"].replace(0, np.nan)
        rep_df["rank"] = rep_df["wins"].rank(ascending=False, method="average")

        records.append(rep_df)

    long_rank_df = pd.concat(records, ignore_index=True)

    summary = (
        long_rank_df.groupby("item_id")
        .agg(
            avg_rank=("rank", "mean"),
            std_rank=("rank", "std"),
            avg_win_rate=("win_rate", "mean"),
            std_win_rate=("win_rate", "std"),
            total_wins=("wins", "sum"),
            total_games=("games", "sum"),
        )
        .reset_index()
    )

    scored = df.copy()
    scored["item_id"] = scored.index
    scored = scored.merge(summary, on="item_id", how="left")
    scored = scored.sort_values(["avg_rank", "avg_win_rate"], ascending=[True, False])

    return scored, long_rank_df


# ======================================================
# Main
# ======================================================
def main():
    df = pd.read_csv(INPUT_FILE)

    scored_base, pairwise_df = evaluate_pairwise(df, n_evals=N_EVALS)
    scored, long_rank_df = aggregate_rankings(scored_base, pairwise_df, n_evals=N_EVALS)

    Path("phase3b-results").mkdir(exist_ok=True)

    scored.to_csv(OUTPUT_FILE, index=False)
    print(f"Saved pairwise-ranked results -> {OUTPUT_FILE}")

    leaderboard = (
        scored.groupby(["model_id", "model_label", "condition"])
        .agg(
            avg_rank=("avg_rank", "mean"),
            avg_win_rate=("avg_win_rate", "mean"),
            std_rank=("avg_rank", "std"),
            std_win_rate=("avg_win_rate", "std"),
            total_wins=("total_wins", "mean"),
            total_games=("total_games", "mean"),
        )
        .sort_values(by=["avg_rank", "avg_win_rate"], ascending=[True, False])
        .reset_index()
    )

    leaderboard.to_csv(LEADERBOARD_FILE, index=False)
    print(f"Saved leaderboard -> {LEADERBOARD_FILE}")

    print("\nTop ranked outputs:")
    print(
        scored[
            [
                "model_label",
                "condition",
                "replicate_id",
                "avg_rank",
                "std_rank",
                "avg_win_rate",
                "total_wins",
                "total_games",
            ]
        ].head(10)
    )

    print("\nLeaderboard:")
    print(
        leaderboard[
            [
                "model_label",
                "condition",
                "avg_rank",
                "avg_win_rate",
                "std_rank",
                "std_win_rate",
            ]
        ].head(10)
    )


if __name__ == "__main__":
    main()

In [ ]:
# ============================================================
# 📊 Phase 3b Visualization — Automation Model Comparison
# ============================================================
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# --- Load leaderboard ---
df = pd.read_csv("phase3b-results/mealplan_automation_pairwise_leaderboard.csv")

# --- Normalize model names to match TASK_MODELS labels ---
model_name_map = {
    "gpt-4.1": "GPT-4.1",
    "deepseek-ai/DeepSeek-V3.1": "DeepSeek-V3.1",
    "o4-mini-2025-04-16": "GPT-O4-Mini",
    "o3-mini-2025-01-31": "GPT-O3-Mini",
    "openai/gpt-oss-120b": "GPT-OSS-120B",
    "anthropic.claude-3-5-sonnet-20240620-v1:0": "Claude-3.5-Sonnet",
    "gpt-5.4-2026-03-05": "GPT-5.4",
    "claude-opus-4-6": "Claude-Opus-4.6",
    "gemini-3.1-pro-preview": "Gemini-3.1-Pro",
}

if "model_label" in df.columns:
    df["plot_model"] = df["model_label"].replace(model_name_map)
elif "model_id" in df.columns:
    df["plot_model"] = df["model_id"].replace(model_name_map)
else:
    raise ValueError("Expected either 'model_label' or 'model_id' column in leaderboard.")

# --- Optional: enforce plotting order to match TASK_MODELS ---
model_order = [
    "GPT-4.1",
    "DeepSeek-V3.1",
    "GPT-O4-Mini",
    "GPT-O3-Mini",
    "GPT-OSS-120B",
    "Claude-3.5-Sonnet",
    "GPT-5.4",
    "Claude-Opus-4.6",
    "Gemini-3.1-Pro",
]

df["plot_model"] = pd.Categorical(
    df["plot_model"],
    categories=model_order,
    ordered=True
)

# --- Convert win rate to percent for plotting ---
df["win_rate_pct"] = df["avg_win_rate"] * 100
df["std_win_rate_pct"] = df["std_win_rate"] * 100 if "std_win_rate" in df.columns else 0.0

# --- Sort for display ---
df = df.sort_values(by="plot_model")

# --- Seaborn style ---
sns.set(style="whitegrid", context="talk", font_scale=1.1)
plt.figure(figsize=(14, 7))

# --- Barplot ---
ax = sns.barplot(
    data=df,
    x="plot_model",
    y="win_rate_pct",
    color="#8172B3",
)

# --- Add custom error bars manually ---
for i, bar in enumerate(ax.patches):
    if i < len(df):
        row = df.iloc[i]
        x = bar.get_x() + bar.get_width() / 2
        y = bar.get_height()
        plt.errorbar(
            x=x,
            y=y,
            yerr=row["std_win_rate_pct"],
            fmt="none",
            ecolor="black",
            elinewidth=1.2,
            capsize=4
        )

# --- Add value labels above each bar ---
for bar, (_, row) in zip(ax.patches, df.iterrows()):
    y = bar.get_height()
    x = bar.get_x() + bar.get_width() / 2
    plt.text(
        x,
        y + 1.0,
        f"{row['win_rate_pct']:.1f}%±{row['std_win_rate_pct']:.1f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

# --- Labels and title ---
plt.title(
    "Automation Model Performance on Meal Plan Task\n"
    "(Mean Pairwise Win Rate ± 1 SD)",
    fontsize=17,
    weight="bold"
)
plt.ylabel("Average Pairwise Win Rate (%)", fontsize=13)
plt.xlabel("Automation Model", fontsize=12)
plt.xticks(rotation=30, ha="right", fontsize=11)
plt.ylim(0, 100)
plt.tight_layout()

# --- Save & show ---
plt.savefig("phase3b_automation_winrate.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import shutil
from google.colab import files
import os

output_folder = "phase3b-outputs"
results_folder = "phase3b-results"
zip_name = "phase3b_experiment_data"

# Create the zip archive
if os.path.exists(output_folder):
    shutil.make_archive(zip_name + "_outputs", 'zip', output_folder)
    print(f"Created {zip_name}_outputs.zip")
    files.download(f"{zip_name}_outputs.zip")
else:
    print(f"Folder '{output_folder}' does not exist.")

if os.path.exists(results_folder):
    shutil.make_archive(zip_name + "_results", 'zip', results_folder)
    print(f"Created {zip_name}_results.zip")
    files.download(f"{zip_name}_results.zip")
else:
    print(f"Folder '{results_folder}' does not exist.")

print("Download process initiated. Please check your browser for the downloads.")